In [6]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, MinMaxScaler, PowerTransformer, OrdinalEncoder
from sklearn.model_selection import train_test_split

In [7]:
%pip install mlflow dagshub

In [8]:
import dagshub
dagshub.init(repo_owner='xlrboi', repo_name='delievery_time_prediction', mlflow=True)

Initialized MLflow to track repo "xlrboi/delievery_time_prediction"

Repository xlrboi/delievery_time_prediction initialized!

In [5]:
import mlflow

In [9]:
mlflow.set_tracking_uri("https://dagshub.com/xlrboi/delievery_time_prediction.mlflow")

In [10]:
mlflow.set_experiment("3 - XGB HP Tuning")

<Experiment: artifact_location='mlflow-artifacts:/0df824ba0f2f4b7db389de7417d05eeb', creation_time=1784396704636, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1784396704636, lifecycle_stage='active', name='3 - XGB HP Tuning', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [11]:
from sklearn import set_config

set_config(transform_output="pandas")

In [12]:
df = pd.read_csv('cleaned_data.csv')
df

,age,ratings,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,time_taken,is_weekend,pickup_time_minutes,order_time_of_day,distance,distance_type
0,37.0,4.9,sunny,high,2,snack,motorcycle,0.0,no,urban,24,1,15.0,morning,3.025149,short
1,34.0,4.5,stormy,jam,2,snack,scooter,1.0,no,metropolitian,33,0,5.0,evening,20.183530,very_long
2,23.0,4.4,sandstorms,low,0,drinks,motorcycle,1.0,no,urban,26,1,15.0,morning,1.552758,short
3,38.0,4.7,sunny,medium,0,buffet,motorcycle,1.0,no,metropolitian,21,0,10.0,evening,7.790401,medium
4,32.0,4.6,cloudy,high,1,snack,scooter,1.0,no,metropolitian,30,1,15.0,afternoon,6.210138,medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37690,35.0,4.2,windy,jam,2,drinks,motorcycle,1.0,no,metropolitian,33,0,10.0,night,16.600272,very_long
37691,30.0,4.8,windy,high,1,meal,motorcycle,0.0,no,metropolitian,32,0,10.0,morning,1.489846,short
37692,30.0,4.9,cloudy,low,1,drinks,scooter,0.0,no,metropolitian,16,0,15.0,night,4.657195,short
37693,20.0,4.7,cloudy,high,0,snack,motorcycle,1.0,no,metropolitian,26,0,5.0,afternoon,6.232393,medium


In [13]:
temp_df = df.copy().dropna()

In [14]:
X = temp_df.drop(columns='time_taken')
y = temp_df['time_taken']

X

,age,ratings,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,is_weekend,pickup_time_minutes,order_time_of_day,distance,distance_type
0,37.0,4.9,sunny,high,2,snack,motorcycle,0.0,no,urban,1,15.0,morning,3.025149,short
1,34.0,4.5,stormy,jam,2,snack,scooter,1.0,no,metropolitian,0,5.0,evening,20.183530,very_long
2,23.0,4.4,sandstorms,low,0,drinks,motorcycle,1.0,no,urban,1,15.0,morning,1.552758,short
3,38.0,4.7,sunny,medium,0,buffet,motorcycle,1.0,no,metropolitian,0,10.0,evening,7.790401,medium
4,32.0,4.6,cloudy,high,1,snack,scooter,1.0,no,metropolitian,1,15.0,afternoon,6.210138,medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37690,35.0,4.2,windy,jam,2,drinks,motorcycle,1.0,no,metropolitian,0,10.0,night,16.600272,very_long
37691,30.0,4.8,windy,high,1,meal,motorcycle,0.0,no,metropolitian,0,10.0,morning,1.489846,short
37692,30.0,4.9,cloudy,low,1,drinks,scooter,0.0,no,metropolitian,0,15.0,night,4.657195,short
37693,20.0,4.7,cloudy,high,0,snack,motorcycle,1.0,no,metropolitian,0,5.0,afternoon,6.232393,medium


In [15]:
y

,time_taken
0,24
1,33
2,26
3,21
4,30
...,...
37690,33
37691,32
37692,16
37693,26


In [16]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [17]:
print("The size of train data is",X_train.shape)
print("The shape of test data is",X_test.shape)

The size of train data is (30156, 15)
The shape of test data is (7539, 15)


In [18]:
pt = PowerTransformer()

y_train_pt = pt.fit_transform(y_train.values.reshape(-1,1))
y_test_pt = pt.transform(y_test.values.reshape(-1,1))

In [19]:
num_cols = ["age","ratings","pickup_time_minutes","distance"]

nominal_cat_cols = ['weather',
                    'type_of_order',
                    'type_of_vehicle',
                    "festival",
                    "city_type",
                    "is_weekend",
                    "order_time_of_day"]

ordinal_cat_cols = ["traffic","distance_type"]

In [20]:
nominal_cat_cols

['weather',
 'type_of_order',
 'type_of_vehicle',
 'festival',
 'city_type',
 'is_weekend',
 'order_time_of_day']

In [21]:
# # features to fill values with mode

# features_to_fill_mode = ['multiple_deliveries','festival','city_type']
# features_to_fill_missing = [col for col in nominal_cat_cols if col not in features_to_fill_mode]

# features_to_fill_missing

In [22]:
# # simple imputer to fill categorical vars with mode

# simple_imputer = ColumnTransformer(transformers=[
#     ("mode_imputer",SimpleImputer(strategy="most_frequent",add_indicator=True),features_to_fill_mode),
#     ("missing_imputer",SimpleImputer(strategy="constant",fill_value="missing",add_indicator=True),features_to_fill_missing)
# ],remainder="passthrough",n_jobs=-1,force_int_remainder_cols=False,verbose_feature_names_out=False)

# simple_imputer

In [23]:
# simple_imputer.fit_transform(X_train)

In [24]:
# simple_imputer.fit_transform(X_train).isna().sum()

In [25]:
# knn imputer

# knn_imputer = KNNImputer(n_neighbors=5)

In [26]:
# generate order for ordinal encoding

traffic_order = ["low","medium","high","jam"]

distance_type_order = ["short","medium","long","very_long"]

In [27]:
# unique categories the ordinal columns

for col in ordinal_cat_cols:
    print(col,X_train[col].unique())

traffic ['jam' 'medium' 'high' 'low']
distance_type ['medium' 'short' 'long' 'very_long']


In [28]:
# build a preprocessor

preprocessor = ColumnTransformer(transformers=[
    ("scale", MinMaxScaler(), num_cols),
    ("nominal_encode", OneHotEncoder(drop="first",handle_unknown="ignore",
                                     sparse_output=False), nominal_cat_cols),
    ("ordinal_encode", OrdinalEncoder(categories=[traffic_order,distance_type_order],
                                      encoded_missing_value=-999,
                                      handle_unknown="use_encoded_value",
                                      unknown_value=-1), ordinal_cat_cols)
],remainder="passthrough",n_jobs=-1,verbose_feature_names_out=False)


preprocessor

ColumnTransformer(n_jobs=-1, remainder='passthrough',
                  transformers=[('scale', MinMaxScaler(),
                                 ['age', 'ratings', 'pickup_time_minutes',
                                  'distance']),
                                ('nominal_encode',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='ignore',
                                               sparse_output=False),
                                 ['weather', 'type_of_order', 'type_of_vehicle',
                                  'festival', 'city_type', 'is_weekend',
                                  'order_time_of_day']),
                                ('ordinal_encode',
                                 OrdinalEncoder(categories=[['low', 'medium',
                                                             'high', 'jam'],
                                                            ['short', 'medium',
                                                             'long',
                                                             'very_long']],
                                                encoded_missing_value=-999,
                                                handle_unknown='use_encoded_value',
                                                unknown_value=-1),
                                 ['traffic', 'distance_type'])],
                  verbose_feature_names_out=False)

In [29]:
processing_pipeline = Pipeline(steps=[
                                # ("simple_imputer",simple_imputer),
                                ("preprocess",preprocessor)
                                # ("knn_imputer",knn_imputer)
                            ])

processing_pipeline

Pipeline(steps=[('preprocess',
                 ColumnTransformer(n_jobs=-1, remainder='passthrough',
                                   transformers=[('scale', MinMaxScaler(),
                                                  ['age', 'ratings',
                                                   'pickup_time_minutes',
                                                   'distance']),
                                                 ('nominal_encode',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['weather', 'type_of_order',
                                                   'type_of_vehicle',
                                                   'festival', 'city_type',
                                                   'is_weekend',
                                                   'order_time_of_day']),
                                                 ('ordinal_encode',
                                                  OrdinalEncoder(categories=[['low',
                                                                              'medium',
                                                                              'high',
                                                                              'jam'],
                                                                             ['short',
                                                                              'medium',
                                                                              'long',
                                                                              'very_long']],
                                                                 encoded_missing_value=-999,
                                                                 handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['traffic',
                                                   'distance_type'])],
                                   verbose_feature_names_out=False))])

In [30]:
X_train_trans = processing_pipeline.fit_transform(X_train)
X_test_trans = processing_pipeline.transform(X_test)

In [31]:
X_train_trans

,age,ratings,pickup_time_minutes,distance,weather_fog,weather_sandstorms,weather_stormy,weather_sunny,weather_windy,type_of_order_drinks,...,city_type_semi-urban,city_type_urban,is_weekend_1,order_time_of_day_evening,order_time_of_day_morning,order_time_of_day_night,traffic,distance_type,vehicle_condition,multiple_deliveries
7204,0.473684,0.56,1.0,0.404165,0.0,0.0,0.0,1.0,0.0,1.0,...,1.0,0.0,0.0,0.0,0.0,1.0,3.0,1.0,0,2.0
20974,1.000000,0.76,0.0,0.154044,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0,1.0
28236,0.473684,0.80,0.5,0.002461,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,2.0,0.0,1,0.0
21635,1.000000,0.92,1.0,0.460411,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,2.0,0,1.0
30780,0.526316,0.76,0.5,0.243676,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16850,0.578947,0.92,0.5,0.451895,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,3.0,2.0,0,0.0
6265,0.052632,1.00,1.0,0.612270,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,1.0,2.0,1,1.0
11284,0.526316,0.92,0.0,0.322877,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1,0.0
860,0.947368,0.96,0.5,0.004486,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0,1.0


In [32]:
%pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 20.9 MB/s eta 0:00:00


In [33]:
from xgboost import XGBRegressor
import optuna

In [35]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import cross_validate
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor
import optuna
import mlflow
import mlflow.sklearn

def objective(trial):
    with mlflow.start_run(nested=True):

        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 1200),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 12),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "gamma": trial.suggest_float("gamma", 0, 5),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-5, 10, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-5, 10, log=True),
            "random_state": 42,
            "device": "cuda",
            "tree_method": "hist",
            "objective": "reg:squarederror",
            "eval_metric": "mae",
            "verbosity": 0,
        }

        model = TransformedTargetRegressor(
            regressor=XGBRegressor(**params),
            transformer=pt
        )

        results = cross_validate(
            estimator=model,
            X=X_train_trans,
            y=y_train,
            cv=5,
            scoring="neg_mean_absolute_error",
            return_train_score=True,
            n_jobs=1
        )

        cv_mae = -results["test_score"].mean()
        train_mae = -results["train_score"].mean()
        cv_std = results["test_score"].std()
        fit_time = results["fit_time"].mean()

        mlflow.log_params(params)
        mlflow.log_metric("cv_mae", cv_mae)
        mlflow.log_metric("train_mae", train_mae)
        mlflow.log_metric("cv_std", cv_std)
        mlflow.log_metric("fit_time", fit_time)

        return cv_mae

study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner()
)

with mlflow.start_run(run_name="XGBoost Hyperparameter Tuning"):

    study.optimize(
        objective,
        n_trials=100,
        n_jobs=1,
        show_progress_bar=True
    )

    mlflow.log_params(study.best_params)
    mlflow.log_metric("best_cv_mae", study.best_value)

[I 2026-07-18 19:22:03,009] A new study created in memory with name: no-name-0ad02056-9798-4ffb-8e3c-d44119f51a2f


  0%|          | 0/100 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but PowerTransformer was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but PowerTransformer was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but PowerTransformer was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but PowerTransformer was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but PowerTransformer was fitted without feature names
  warnings.warn(


🏃 View run useful-ox-722 at: https://dagshub.com/xlrboi/delievery_time_prediction.mlflow/#/experiments/3/runs/15c335fb8c764c289e948e1c911a3a24
🧪 View experiment at: https://dagshub.com/xlrboi/delievery_time_prediction.mlflow/#/experiments/3
[I 2026-07-18 19:22:11,248] Trial 0 finished with value: 3.1753173828125 and parameters: {'n_estimators': 574, 'learning_rate': 0.2536999076681772, 'max_depth': 10, 'min_child_weight': 6, 'gamma': 0.7800932022121826, 'subsample': 0.662397808134481, 'colsample_bytree': 0.6232334448672797, 'reg_alpha': 1.574189004745663, 'reg_lambda': 0.040428727350273294}. Best is trial 0 with value: 3.1753173828125.


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but PowerTransformer was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but PowerTransformer was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but PowerTransformer was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but PowerTransformer was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but PowerTransformer was fitted without feature names
  warnings.warn(


...

🏃 View run valuable-bat-659 at: https://dagshub.com/xlrboi/delievery_time_prediction.mlflow/#/experiments/3/runs/f02f877b32ba431e823fe18f566bc59c
🧪 View experiment at: https://dagshub.com/xlrboi/delievery_time_prediction.mlflow/#/experiments/3
[I 2026-07-18 19:45:32,826] Trial 97 finished with value: 3.023421049118042 and parameters: {'n_estimators': 856, 'learning_rate': 0.011804738897525, 'max_depth': 12, 'min_child_weight': 6, 'gamma': 0.6035875191188472, 'subsample': 0.9028989534142778, 'colsample_bytree': 0.9791480707107688, 'reg_alpha': 0.00020972186331087944, 'reg_lambda': 0.0033654731421510226}. Best is trial 97 with value: 3.023421049118042.


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but PowerTransformer was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but PowerTransformer was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but PowerTransformer was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but PowerTransformer was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but PowerTransformer was fitted without feature names
  warnings.warn(


🏃 View run dazzling-moth-181 at: https://dagshub.com/xlrboi/delievery_time_prediction.mlflow/#/experiments/3/runs/ed4f4e1b352a493bacf2c9a5637708d5
🧪 View experiment at: https://dagshub.com/xlrboi/delievery_time_prediction.mlflow/#/experiments/3
[I 2026-07-18 19:45:50,011] Trial 98 finished with value: 3.0242630004882813 and parameters: {'n_estimators': 876, 'learning_rate': 0.010083080300631106, 'max_depth': 11, 'min_child_weight': 6, 'gamma': 0.5113753636039032, 'subsample': 0.9411287021126775, 'colsample_bytree': 0.9819661262405106, 'reg_alpha': 1.272561662934443e-05, 'reg_lambda': 0.0015304635916392488}. Best is trial 97 with value: 3.023421049118042.


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but PowerTransformer was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but PowerTransformer was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but PowerTransformer was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but PowerTransformer was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but PowerTransformer was fitted without feature names
  warnings.warn(


🏃 View run secretive-koi-997 at: https://dagshub.com/xlrboi/delievery_time_prediction.mlflow/#/experiments/3/runs/d6facf1636bc4c3785e233301af57011
🧪 View experiment at: https://dagshub.com/xlrboi/delievery_time_prediction.mlflow/#/experiments/3
[I 2026-07-18 19:46:04,725] Trial 99 finished with value: 3.35311222076416 and parameters: {'n_estimators': 954, 'learning_rate': 0.01067006018460292, 'max_depth': 4, 'min_child_weight': 5, 'gamma': 0.4945417796810887, 'subsample': 0.9072338012075106, 'colsample_bytree': 0.9370345103698375, 'reg_alpha': 1.130459826091688e-05, 'reg_lambda': 0.0016642727278612995}. Best is trial 97 with value: 3.023421049118042.
🏃 View run XGBoost Hyperparameter Tuning at: https://dagshub.com/xlrboi/delievery_time_prediction.mlflow/#/experiments/3/runs/f4f2df0159db4b58aa70dbd7231dad90
🧪 View experiment at: https://dagshub.com/xlrboi/delievery_time_prediction.mlflow/#/experiments/3


In [36]:
best_xgb = XGBRegressor(
        **study.best_params,
        random_state=42,
        device="cuda",
        tree_method="hist",
        objective="reg:squarederror",
        eval_metric="mae",
        verbosity=0
    )
final_model = TransformedTargetRegressor(
        regressor=best_xgb,
        transformer=pt
    )

final_model.fit(X_train_trans, y_train)

y_pred_train = final_model.predict(X_train_trans)
y_pred_test = final_model.predict(X_test_trans)

scores = cross_validate(
        estimator=final_model,
        X=X_train_trans,
        y=y_train,
        cv=5,
        scoring="neg_mean_absolute_error",
        return_train_score=True,
        n_jobs=1
    )

mlflow.log_metric("train_mae", mean_absolute_error(y_train, y_pred_train))
mlflow.log_metric("test_mae", mean_absolute_error(y_test, y_pred_test))
mlflow.log_metric("train_r2", r2_score(y_train, y_pred_train))
mlflow.log_metric("test_r2", r2_score(y_test, y_pred_test))
mlflow.log_metric("final_cv_mae", -scores["test_score"].mean())

mlflow.sklearn.log_model(final_model, name="model",skops_trusted_types=[
        "xgboost.core.Booster",
        "xgboost.sklearn.XGBRegressor",
    ],)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but PowerTransformer was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but PowerTransformer was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but PowerTransformer was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but PowerTransformer was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but PowerTransformer was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature name

In [40]:
study.best_params

{'n_estimators': 856,
 'learning_rate': 0.011804738897525,
 'max_depth': 12,
 'min_child_weight': 6,
 'gamma': 0.6035875191188472,
 'subsample': 0.9028989534142778,
 'colsample_bytree': 0.9791480707107688,
 'reg_alpha': 0.00020972186331087944,
 'reg_lambda': 0.0033654731421510226}

In [41]:
study.best_value

3.023421049118042

In [37]:
optuna.visualization.plot_optimization_history(study)

In [38]:
optuna.visualization.plot_param_importances(study)

In [39]:
optuna.visualization.plot_slice(study)